In [1]:
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")


Mounted at /content/drive
Google Drive mounted successfully.


In [3]:
# ============================================================
# CELL 2 — IMPORTS AND COLAB CONFIGURATION
# ============================================================

import sys
import os
import re
import time
import pickle
import random
import shutil
from pathlib import Path
from collections import deque

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from tqdm.auto import tqdm


# -----------------------------
# Reproducibility
# -----------------------------
SEED = 30
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# -----------------------------
# Google Drive / processed data
# -----------------------------
MYDRIVE_ROOT = Path("/content/drive/MyDrive")

# IMPORTANT:
# Set this to the path of the SHORTCUT itself as it appears in My Drive.
# It can also include parent folders, for example:
#     "GSTCAN/Processed Shortcut"
#
# Do not add the .pkl filenames here.
PROCESSED_SHORTCUT_PATH = "UrProcessed"

PROCESSED_DIR = MYDRIVE_ROOT / PROCESSED_SHORTCUT_PATH


# -----------------------------
# Output directories on Drive
# -----------------------------
OUTPUT_ROOT = MYDRIVE_ROOT / "GSTCAN_ShiftGCN_Colab_Results"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints_shiftgcn_fullseq"
RESULTS_DIR = OUTPUT_ROOT / "results_shiftgcn_fullseq"

for directory in [
    OUTPUT_ROOT,
    CHECKPOINT_DIR,
    RESULTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# -----------------------------
# GSTCAN model/training
# -----------------------------
FINAL_EPOCHS = 60
BATCH_SIZE = 12
LEARNING_RATE = 0.0005
DROPOUT = 0.5
TEMPORAL_KERNEL = 9
GSTCAN_CHANNELS = [64, 64, 128, 128, 256, 256]
NUM_CLASSES = 2
NUM_JOINTS = 13
VALIDATION_RATIO = 0.1
TOP_K_CHECKPOINTS = 3
GRAPH_OPERATOR = "Shift-GCN-style"

print("Configuration loaded.")


Configuration loaded.


In [ ]:
# ============================================================
# CELL 3 — VERIFY COLAB ENVIRONMENT + PROCESSED DATA
# ============================================================

print("Python       :", sys.version)
print("PyTorch      :", torch.__version__)
print("PyTorch CUDA :", torch.version.cuda)
print("CUDA         :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "CUDA is not available. In Colab, select Runtime -> Change runtime type -> GPU."
    )

assert MYDRIVE_ROOT.exists(), (
    f"Google Drive root does not exist: {MYDRIVE_ROOT}"
)

assert PROCESSED_DIR.exists(), (
    "Processed shortcut/folder does not exist:\n"
    f"{PROCESSED_DIR}\n\n"
    "Set PROCESSED_SHORTCUT_PATH to the exact shortcut path inside My Drive."
)

assert PROCESSED_DIR.is_dir(), (
    f"Processed path is not a directory: {PROCESSED_DIR}"
)

print("Processed directory:", PROCESSED_DIR)
print("Environment and path verification PASSED.")


# ============================================================
# CELL 4 — DISCOVER + VALIDATE EXISTING PROCESSED PKL FILES
# ============================================================

processed_files = sorted(
    p for p in PROCESSED_DIR.glob("*.pkl")
    if p.is_file()
)

assert processed_files, (
    f"No .pkl files found directly inside: {PROCESSED_DIR}"
)

sequence_records = []

required_keys = {
    "sequence",
    "label",
    "motion",
}

for file_path in processed_files:
    try:
        with open(file_path, "rb") as f:
            data = pickle.load(f)
    except Exception as exc:
        raise RuntimeError(
            f"Could not load pickle file: {file_path}"
        ) from exc

    missing_keys = required_keys - set(data.keys())
    assert not missing_keys, (
        f"Missing required keys in {file_path}: {sorted(missing_keys)}"
    )

    sequence_name = str(data["sequence"])
    label = int(data["label"])

    assert label in (0, 1), (
        f"Invalid label in {file_path}: {label}. Expected 0 or 1."
    )

    motion = np.asarray(data["motion"], dtype=np.float32)

    assert motion.ndim == 3, (
        f"Invalid motion dimensions in {file_path}: {motion.shape}"
    )
    assert motion.shape[1:] == (13, 2), (
        f"Expected motion shape [T,13,2] in {file_path}, got {motion.shape}"
    )
    assert np.isfinite(motion).all(), (
        f"Non-finite motion values found in {file_path}"
    )

    sequence_records.append(
        {
            "sequence": sequence_name,
            "path": file_path,
            "label": label,
            "num_frames": int(motion.shape[0]),
        }
    )

sequence_names = [
    record["sequence"]
    for record in sequence_records
]

assert len(sequence_names) == len(set(sequence_names)), (
    "Duplicate sequence names were found in the processed PKL files."
)

fall_count = int(np.sum([
    record["label"] == 1
    for record in sequence_records
]))

adl_count = int(np.sum([
    record["label"] == 0
    for record in sequence_records
]))

print("============================================================")
print("PROCESSED DATASET DISCOVERY")
print("============================================================")
print("Processed directory:", PROCESSED_DIR)
print("PKL files found    :", len(processed_files))
print("Fall sequences     :", fall_count)
print("ADL sequences      :", adl_count)

for record in sorted(sequence_records, key=lambda x: x["sequence"].lower()):
    label_name = "Fall" if record["label"] == 1 else "ADL"
    print(
        f"{record['sequence']:<40} | "
        f"{label_name:<5} | "
        f"frames={record['num_frames']:>6} | "
        f"{record['path']}"
    )

assert fall_count >= 3 and adl_count >= 3, (
    "At least 3 Fall and 3 ADL sequences are required for 3-fold stratified CV."
)

print("\nProcessed PKL validation PASSED.")


# ============================================================
# CELL 10 — 13-JOINT DEFINITION
# ============================================================

JOINT_NAMES = [
    "neck",
    "left_shoulder",
    "left_elbow",
    "left_wrist",
    "right_shoulder",
    "right_elbow",
    "right_wrist",
    "left_hip",
    "left_knee",
    "left_ankle",
    "right_hip",
    "right_knee",
    "right_ankle",
]

EDGES_13 = [
    (0, 1),
    (1, 2),
    (2, 3),
    (0, 4),
    (4, 5),
    (5, 6),
    (0, 7),
    (7, 8),
    (8, 9),
    (0, 10),
    (10, 11),
    (11, 12),
    (1, 7),
    (4, 10),
]

assert len(JOINT_NAMES) == 13
assert len(EDGES_13) == 14

print("13-joint definition ready.")
print("Joints:", len(JOINT_NAMES))
print("Edges :", len(EDGES_13))


# CELL 11 — EXACT REPRODUCIBLE GRAPH CONSTRUCTION
# ============================================================

# The paper describes:
#   - graph G=(V,E)
#   - natural body-joint connections
#   - self-connections
#   - root / centripetal / centrifugal spatial configurations
#   - normalization using D^(-1/2) A D^(-1/2)
#
# The 13 selected joints are used here because the paper explicitly says
# that 13 body landmarks were selected for the GSTCAN input after pose
# estimation.

ROOT_NODE = 0  # neck


def shortest_hop_distances(num_nodes: int, edges, root: int) -> np.ndarray:
    graph = [[] for _ in range(num_nodes)]

    for i, j in edges:
        graph[i].append(j)
        graph[j].append(i)

    distances = np.full(num_nodes, np.inf, dtype=np.float32)
    distances[root] = 0.0
    queue = deque([root])

    while queue:
        node = queue.popleft()
        for neighbor in graph[node]:
            if np.isinf(distances[neighbor]):
                distances[neighbor] = distances[node] + 1.0
                queue.append(neighbor)

    return distances


def build_spatial_partitions(num_nodes: int, edges, root: int):
    hop = shortest_hop_distances(num_nodes, edges, root)
    partitions = np.zeros((3, num_nodes, num_nodes), dtype=np.float32)

    # Root partition receives every self-connection.
    for i in range(num_nodes):
        partitions[0, i, i] = 1.0

    # For each undirected anatomical edge, add BOTH directed directions.
    for i, j in edges:
        # i -> j
        if hop[j] == hop[i]:
            partition_ij = 0
        elif hop[j] < hop[i]:
            partition_ij = 1       # centripetal: toward root
        else:
            partition_ij = 2       # centrifugal: away from root

        partitions[partition_ij, i, j] = 1.0

        # j -> i
        if hop[i] == hop[j]:
            partition_ji = 0
        elif hop[i] < hop[j]:
            partition_ji = 1
        else:
            partition_ji = 2

        partitions[partition_ji, j, i] = 1.0

    return hop, partitions


HOP_DISTANCE, A_PARTITIONS = build_spatial_partitions(
    NUM_JOINTS,
    EDGES_13,
    ROOT_NODE,
)


# Normalize each spatial partition as:
#     A_norm = D^(-1/2) A D^(-1/2)
# with D built from the outgoing degree of that partition.
A_PARTITIONS_NORM = np.zeros_like(A_PARTITIONS)

for k in range(3):
    A_k = A_PARTITIONS[k]
    degree = np.sum(A_k, axis=1)

    degree_inv_sqrt = np.zeros_like(degree)
    valid = degree > 0
    degree_inv_sqrt[valid] = 1.0 / np.sqrt(degree[valid])

    D_inv_sqrt = np.diag(degree_inv_sqrt)
    A_PARTITIONS_NORM[k] = (
        D_inv_sqrt @ A_k @ D_inv_sqrt
    )

assert A_PARTITIONS.shape == (3, 13, 13)
assert A_PARTITIONS_NORM.shape == (3, 13, 13)
assert np.all(np.diag(A_PARTITIONS[0]) == 1.0)
assert np.sum(A_PARTITIONS[1]) > 0
assert np.sum(A_PARTITIONS[2]) > 0
assert np.isfinite(A_PARTITIONS_NORM).all()
assert np.all(np.sum(A_PARTITIONS, axis=0) <= 1.0)

A_partition_tensor = torch.tensor(
    A_PARTITIONS_NORM,
    dtype=torch.float32,
)

print("Graph partitions ready:", tuple(A_partition_tensor.shape))
print("Root node:", JOINT_NAMES[ROOT_NODE])


# ============================================================


# CELL 12 — GSTCAN CHANNEL ATTENTION
# ============================================================

class GSTCANChannelAttention(nn.Module):
    def __init__(self, channels: int):
        super().__init__()

        hidden = max(1, channels // 4)

        self.fc1 = nn.Linear(channels, hidden)
        self.bn = nn.BatchNorm1d(hidden)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(hidden, channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x = [B,C,T,V]
        pooled = x.mean(dim=(2, 3))
        attention = self.fc1(pooled)
        attention = self.bn(attention)
        attention = self.relu(attention)
        attention = self.fc2(attention)
        attention = self.sigmoid(attention)
        attention = attention.view(x.shape[0], x.shape[1], 1, 1)
        return x * attention


# ============================================================


# CELL 13 — SHIFT-GCN-STYLE GRAPH CONVOLUTION
# ============================================================

class ShiftGraphConv(nn.Module):
    """
    Lightweight Shift-GCN-style spatial operator.

    The original GSTCAN block uses three independent 1x1 projections
    followed by three adjacency-matrix multiplications. This version
    replaces those operations with:
      1) fixed channel-wise graph shifts derived from the same
         root / centripetal / centrifugal graph partitions
      2) one shared 1x1 projection

    The graph topology therefore remains the same, but the expensive
    learnable graph projection is performed only once per unit.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        adjacency_partitions,
    ):
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_partitions = 3
        self.num_nodes = int(
            np.asarray(adjacency_partitions).shape[-1]
        )

        if torch.is_tensor(adjacency_partitions):
            A = adjacency_partitions.detach().cpu().numpy()
        else:
            A = np.asarray(adjacency_partitions)

        assert A.shape == (
            self.num_partitions,
            self.num_nodes,
            self.num_nodes,
        )

        # --------------------------------------------------------
        # Construct one source-joint index for every
        # (input channel, target joint).
        #
        # Channel groups cycle through:
        #   0 = root / self
        #   1 = centripetal / toward root
        #   2 = centrifugal / away from root
        #
        # When a target has multiple valid neighbors in a
        # partition, different channels cycle through those
        # neighbors. This keeps branches such as the neck's
        # multiple children represented.
        # --------------------------------------------------------
        shift_indices = np.zeros(
            (in_channels, self.num_nodes),
            dtype=np.int64,
        )

        for channel in range(in_channels):
            partition = channel % self.num_partitions

            for target_node in range(self.num_nodes):
                candidates = np.flatnonzero(
                    A[partition, :, target_node] > 0
                )

                if candidates.size == 0:
                    candidates = np.array(
                        [target_node],
                        dtype=np.int64,
                    )

                candidate_index = (
                    (channel // self.num_partitions)
                    % candidates.size
                )

                shift_indices[channel, target_node] = (
                    candidates[candidate_index]
                )

        self.register_buffer(
            "shift_indices",
            torch.as_tensor(
                shift_indices,
                dtype=torch.long,
            ),
        )

        # Small learnable channel/joint mask.
        self.feature_mask = nn.Parameter(
            torch.ones(
                1,
                in_channels,
                1,
                self.num_nodes,
                dtype=torch.float32,
            )
        )

        # One learnable projection instead of the original
        # three partition-specific 1x1 projections.
        self.proj = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x = [B,C_in,T,V]

        batch_size, channels, time_steps, num_nodes = x.shape

        assert channels == self.in_channels, (
            f"Expected {self.in_channels} input channels, "
            f"got {channels}."
        )

        assert num_nodes == self.num_nodes, (
            f"Expected {self.num_nodes} joints, "
            f"got {num_nodes}."
        )

        indices = self.shift_indices.view(
            1,
            channels,
            1,
            num_nodes,
        ).expand(
            batch_size,
            channels,
            time_steps,
            num_nodes,
        )

        # Fixed graph shift.
        shifted = torch.gather(
            x,
            dim=3,
            index=indices,
        )

        shifted = shifted * self.feature_mask

        # Single 1x1 pointwise projection.
        return self.proj(shifted)


# ============================================================


# CELL 14 — GSTCAN UNIT
# ============================================================

class GSTCANUnit(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        adjacency_partitions,
        temporal_kernel: int = TEMPORAL_KERNEL,
        dropout: float = DROPOUT,
    ):
        super().__init__()

        self.gcn = ShiftGraphConv(
            in_channels,
            out_channels,
            adjacency_partitions,
        )

        padding = temporal_kernel // 2

        self.tcn = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=(temporal_kernel, 1),
            padding=(padding, 0),
            bias=False,
        )

        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.attention = GSTCANChannelAttention(out_channels)
        self.dropout = nn.Dropout(p=dropout)

        self.out_channels = in_channels + out_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gcn = self.relu(self.gcn(x))
        tcn = self.tcn(gcn)
        tcn = self.relu(self.bn(tcn))
        refined = self.attention(tcn)

        # Paper pseudocode specifies concatenation of CAM with incoming GCN input.
        out = torch.cat([x, refined], dim=1)
        return self.dropout(out)


# ============================================================


# CELL 15 — COMPLETE GSTCAN MODEL
# ============================================================

class GSTCAN(nn.Module):
    def __init__(
        self,
        adjacency_partitions,
        input_channels: int = 2,
        num_classes: int = NUM_CLASSES,
        dropout: float = DROPOUT,
    ):
        super().__init__()

        self.input_bn = nn.BatchNorm2d(input_channels)

        self.units = nn.ModuleList()
        current_channels = input_channels

        for out_channels in GSTCAN_CHANNELS:
            unit = GSTCANUnit(
                in_channels=current_channels,
                out_channels=out_channels,
                adjacency_partitions=adjacency_partitions,
                temporal_kernel=TEMPORAL_KERNEL,
                dropout=dropout,
            )
            self.units.append(unit)
            current_channels += out_channels

        self.final_channels = current_channels
        self.classifier = nn.Linear(
            self.final_channels,
            num_classes,
        )

    def forward(self, x: torch.Tensor, temporal_mask: torch.Tensor) -> torch.Tensor:
        # x = [B,2,T,13]
        # temporal_mask = [B,1,T,1]
        x = self.input_bn(x)
        x = x * temporal_mask

        for unit in self.units:
            x = unit(x)
            # Prevent padded temporal positions from contributing to later
            # GSTCAN units and the final pooled representation.
            x = x * temporal_mask

        masked_x = x * temporal_mask
        pooled_sum = masked_x.sum(dim=(2, 3))

        valid_time_steps = temporal_mask[:, 0, :, 0].sum(dim=1)
        valid_elements = torch.clamp(
            valid_time_steps * x.shape[3],
            min=1.0,
        ).unsqueeze(1)

        pooled = pooled_sum / valid_elements
        return self.classifier(pooled)


# ============================================================


# CELL 15A — SHIFT-GCN PARAMETER SUMMARY
# ============================================================

def count_trainable_parameters(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


_complexity_model = GSTCAN(
    adjacency_partitions=A_partition_tensor,
    input_channels=2,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT,
)

print("============================================================")
print("SHIFT-GCN MODEL COMPLEXITY")
print("============================================================")
print(
    "Trainable parameters:",
    f"{count_trainable_parameters(_complexity_model):,}",
)
print("Graph operator      :", GRAPH_OPERATOR)
print(
    "GCN projection      :",
    "single 1x1 Conv2d after fixed graph shifts",
)
print(
    "Expected GCN-only parameter reduction vs the old "
    "3-branch GCN: approximately 66.7%.",
)
print(
    "The total model reduction is smaller because the TCN, "
    "channel attention, classifier, and other layers are unchanged.",
)

del _complexity_model

# ============================================================


# CELL 16 — FULL-SEQUENCE DATASET / COLLATE
# ============================================================

class ProcessedSequenceDataset(Dataset):
    """One dataset item = one complete pre-processed sequence from a PKL file.

    The PKL filename is NOT assumed to match the internal sequence name.
    This is important because Google Drive may rename files to names such as
    "Copy of adl-01.pkl" or "Copy of video (1).pkl", while the sequence
    stored inside the PKL is "adl-01" or "video (1)".
    """

    def __init__(self, sequence_names, sequence_records):
        self.samples = []

        path_by_sequence = {
            record["sequence"]: Path(record["path"])
            for record in sequence_records
        }

        for sequence_name in sequence_names:
            if sequence_name not in path_by_sequence:
                raise FileNotFoundError(
                    f"No PKL path found for sequence: {sequence_name}"
                )

            file_path = path_by_sequence[sequence_name]

            if not file_path.exists():
                raise FileNotFoundError(
                    f"PKL file does not exist: {file_path}"
                )

            with open(file_path, "rb") as f:
                data = pickle.load(f)

            assert str(data["sequence"]) == sequence_name, (
                f"Sequence mismatch: requested '{sequence_name}', "
                f"but PKL contains '{data['sequence']}'"
            )

            motion = np.asarray(data["motion"], dtype=np.float32)

            assert motion.ndim == 3
            assert motion.shape[1:] == (13, 2)
            assert np.isfinite(motion).all()

            # [T,13,2] -> [2,T,13]
            motion = np.transpose(
                motion,
                (2, 0, 1),
            ).astype(np.float32)

            self.samples.append(
                {
                    "sequence": sequence_name,
                    "motion": motion,
                    "label": int(data["label"]),
                    "path": file_path,
                }
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        item = self.samples[index]

        x = torch.from_numpy(item["motion"])
        y = torch.tensor(
            item["label"],
            dtype=torch.long,
        )

        return x, y, item["sequence"]

def full_sequence_collate(batch):
    xs = [item[0] for item in batch]
    ys = [item[1] for item in batch]
    names = [item[2] for item in batch]

    batch_size = len(xs)
    channels = xs[0].shape[0]
    num_nodes = xs[0].shape[2]
    max_T = max(x.shape[1] for x in xs)

    X = torch.zeros(
        batch_size,
        channels,
        max_T,
        num_nodes,
        dtype=torch.float32,
    )

    temporal_mask = torch.zeros(
        batch_size,
        1,
        max_T,
        1,
        dtype=torch.float32,
    )

    for i, x in enumerate(xs):
        T = x.shape[1]
        X[i, :, :T, :] = x
        temporal_mask[i, 0, :T, 0] = 1.0

    Y = torch.stack(ys)
    return X, Y, temporal_mask, names


# ============================================================


# ============================================================


# CELL 17 — SEQUENCE-LEVEL 3-FOLD SPLIT
# ============================================================

# This is created only after all processed sequences exist.
# No frame-level split is used, so frames from one video cannot cross folds.
# The exact authors' fold membership is not published in the paper.

# IMPORTANT:
# Use the sequence name stored INSIDE each PKL file.
# Do NOT use the PKL filename, because Google Drive may rename files
# to names such as "Copy of adl-01.pkl".
sequence_names = [
    record["sequence"]
    for record in sequence_records
]

assert len(sequence_names) == len(processed_files), (
    f"Expected {len(processed_files)} sequence names, "
    f"found {len(sequence_names)}"
)

assert len(sequence_names) == len(set(sequence_names)), (
    "Duplicate internal sequence names were found."
)

label_by_sequence = {
    record["sequence"]: int(record["label"])
    for record in sequence_records
}

labels = np.array(
    [
        label_by_sequence[name]
        for name in sequence_names
    ],
    dtype=np.int64,
)

assert len(labels) == len(sequence_names)

splitter = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED,
)

folds = []

for fold_id, (train_idx, test_idx) in enumerate(
    splitter.split(sequence_names, labels),
    start=1,
):
    folds.append(
        {
            "fold": fold_id,
            "train_sequences": [sequence_names[i] for i in train_idx],
            "test_sequences": [sequence_names[i] for i in test_idx],
        }
    )

print("Sequence-level 3-fold split created.")

for fold in folds:
    train_labels = [
        label_by_sequence[n]
        for n in fold["train_sequences"]
    ]

    test_labels = [
        label_by_sequence[n]
        for n in fold["test_sequences"]
    ]

    print(
        f"Fold {fold['fold']}: "
        f"train={len(fold['train_sequences'])} "
        f"(fall={sum(train_labels)}, ADL={len(train_labels)-sum(train_labels)}), "
        f"test={len(fold['test_sequences'])} "
        f"(fall={sum(test_labels)}, ADL={len(test_labels)-sum(test_labels)})"
    )



# ============================================================


# CELL 18 — VALIDATION / TEST EVALUATION FUNCTION
# ============================================================

def evaluate_sequence_model(model, loader, device, criterion):
    model.eval()

    running_loss = 0.0
    all_true = []
    all_pred = []

    start = time.perf_counter()

    with torch.no_grad():
        for X, Y, mask, _ in loader:
            X = X.to(device)
            Y = Y.to(device)
            mask = mask.to(device)

            # Zero padded temporal positions.
            X = X * mask

            logits = model(X, mask)
            loss = criterion(logits, Y)

            running_loss += loss.item() * X.size(0)

            predictions = torch.argmax(logits, dim=1)

            all_true.extend(Y.cpu().numpy().tolist())
            all_pred.extend(predictions.cpu().numpy().tolist())

    elapsed = time.perf_counter() - start

    y_true = np.asarray(all_true, dtype=np.int64)
    y_pred = np.asarray(all_pred, dtype=np.int64)

    total = len(y_true)
    average_loss = running_loss / total if total else 0.0

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    return {
        "loss": float(average_loss),
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "precision": float(
            precision_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "confusion_matrix": cm.tolist(),
        "samples": int(total),
        "inference_seconds": float(elapsed),
    }

# ============================================================


# CELL 19 — FINAL 3-FOLD × 100-EPOCH TRAINING
#             TOP-3 CHECKPOINTS BY VALIDATION LOSS
# ============================================================

DEVICE_TORCH = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

criterion = nn.CrossEntropyLoss()

fold_results = []

# Keep the three epochs with the lowest validation loss
TOP_K_CHECKPOINTS = 3


for fold_info in folds:

    fold_id = fold_info["fold"]

    print("\n========================================")
    print(f"STARTING FOLD {fold_id}")
    print("========================================")

    # --------------------------------------------------------
    # OUTER FOLD
    # --------------------------------------------------------

    outer_train_sequences = fold_info["train_sequences"]
    test_sequences = fold_info["test_sequences"]

    outer_train_labels = np.array(
        [
            label_by_sequence[name]
            for name in outer_train_sequences
        ],
        dtype=np.int64,
    )

    # --------------------------------------------------------
    # INNER TRAIN / VALIDATION SPLIT
    # --------------------------------------------------------

    train_sequences, val_sequences = train_test_split(
        outer_train_sequences,
        test_size=VALIDATION_RATIO,
        random_state=SEED,
        stratify=outer_train_labels,
    )

    print(
        f"Fold {fold_id} split:"
        f" train={len(train_sequences)}"
        f" | validation={len(val_sequences)}"
        f" | test={len(test_sequences)}"
    )

    train_labels = [
        label_by_sequence[name]
        for name in train_sequences
    ]

    val_labels = [
        label_by_sequence[name]
        for name in val_sequences
    ]

    test_labels = [
        label_by_sequence[name]
        for name in test_sequences
    ]

    print(
        f"Train      -> fall={sum(train_labels)}, "
        f"ADL={len(train_labels) - sum(train_labels)}"
    )

    print(
        f"Validation -> fall={sum(val_labels)}, "
        f"ADL={len(val_labels) - sum(val_labels)}"
    )

    print(
        f"Test       -> fall={sum(test_labels)}, "
        f"ADL={len(test_labels) - sum(test_labels)}"
    )

    # --------------------------------------------------------
    # DATASETS
    # --------------------------------------------------------

    train_dataset = ProcessedSequenceDataset(
        train_sequences,
        sequence_records,
    )

    val_dataset = ProcessedSequenceDataset(
        val_sequences,
        sequence_records,
    )

    test_dataset = ProcessedSequenceDataset(
        test_sequences,
        sequence_records,
    )

    # --------------------------------------------------------
    # DATALOADERS
    # --------------------------------------------------------

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        collate_fn=full_sequence_collate,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        collate_fn=full_sequence_collate,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        collate_fn=full_sequence_collate,
    )

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model = GSTCAN(
        adjacency_partitions=A_partition_tensor,
        input_channels=2,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT,
    ).to(DEVICE_TORCH)

    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    optimizer = torch.optim.RMSprop(
        model.parameters(),
        lr=LEARNING_RATE,
    )

    # --------------------------------------------------------
    # HISTORY
    # --------------------------------------------------------

    history = {
        "train_loss": [],
        "train_accuracy": [],

        "val_loss": [],
        "val_accuracy": [],
        "val_precision": [],
        "val_recall": [],
        "val_f1": [],
        "val_inference_seconds": [],

        "test_loss": [],
        "test_accuracy": [],
        "test_precision": [],
        "test_recall": [],
        "test_f1": [],
        "test_inference_seconds": [],
    }

    # --------------------------------------------------------
    # TOP-3 CHECKPOINT TRACKING
    # --------------------------------------------------------

    # Each item contains the epoch and its validation loss.
    # Sorted from best (lowest loss) to worst.
    top_checkpoints = []

    fold_start = time.perf_counter()

    # Remove stale top-3 / epoch checkpoint files from a previous run.
    for stale_path in CHECKPOINT_DIR.glob(
        f"gstcan_fold{fold_id}_epoch*.pth"
    ):
        stale_path.unlink()

    for stale_path in CHECKPOINT_DIR.glob(
        f"gstcan_fold{fold_id}_top*.pth"
    ):
        stale_path.unlink()

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    for epoch in range(1, FINAL_EPOCHS + 1):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        # ----------------------------------------------------
        # TRAINING BATCHES
        # ----------------------------------------------------

        for X, Y, mask, _ in train_loader:

            X = X.to(DEVICE_TORCH)
            Y = Y.to(DEVICE_TORCH)
            mask = mask.to(DEVICE_TORCH)

            X = X * mask

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                X,
                mask,
            )

            loss = criterion(
                logits,
                Y,
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    "Non-finite training loss."
                )

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item() * X.size(0)
            )

            predictions = torch.argmax(
                logits,
                dim=1,
            )

            correct += int(
                (predictions == Y).sum()
            )

            total += Y.size(0)

        train_loss = (
            running_loss / total
            if total
            else 0.0
        )

        train_accuracy = (
            correct / total
            if total
            else 0.0
        )

        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------
        # IMPORTANT:
        # The validation set is used for checkpoint selection.
        # The test set is NOT touched here.

        val_metrics = evaluate_sequence_model(
            model,
            val_loader,
            DEVICE_TORCH,
            criterion,
        )

        history["train_loss"].append(
            float(train_loss)
        )

        history["train_accuracy"].append(
            float(train_accuracy)
        )

        history["val_loss"].append(
            float(val_metrics["loss"])
        )

        history["val_accuracy"].append(
            float(val_metrics["accuracy"])
        )

        history["val_precision"].append(
            float(val_metrics["precision"])
        )

        history["val_recall"].append(
            float(val_metrics["recall"])
        )

        history["val_f1"].append(
            float(val_metrics["f1"])
        )

        history["val_inference_seconds"].append(
            float(
                val_metrics["inference_seconds"]
            )
        )

        current_val_loss = float(
            val_metrics["loss"]
        )

        # ----------------------------------------------------
        # LATEST CHECKPOINT
        # ----------------------------------------------------
        # Keep a latest checkpoint as well for debugging/resume.

        latest_checkpoint = {
            "fold": fold_id,
            "epoch": epoch,
            "validation_loss": current_val_loss,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "history": history,

            "config": {
                "epochs": FINAL_EPOCHS,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "dropout": DROPOUT,
                "temporal_kernel": TEMPORAL_KERNEL,
                "gstcan_channels": GSTCAN_CHANNELS,
                "gstcan_units": len(GSTCAN_CHANNELS),
                "validation_ratio": VALIDATION_RATIO,
                "seed": SEED,
                "top_k_checkpoints": TOP_K_CHECKPOINTS,
                "graph_operator": GRAPH_OPERATOR,
                "graph_gcn_projection": "single_1x1_after_fixed_shift",
            },

            "train_sequences": train_sequences,
            "validation_sequences": val_sequences,
            "test_sequences": test_sequences,
        }

        torch.save(
            latest_checkpoint,
            CHECKPOINT_DIR
            / f"gstcan_fold{fold_id}_latest.pth",
        )

        # ----------------------------------------------------
        # UPDATE TOP-3 CHECKPOINT LIST
        # ----------------------------------------------------

        previous_top_epochs = {
            item["epoch"]
            for item in top_checkpoints
        }

        top_checkpoints.append(
            {
                "epoch": epoch,
                "val_loss": current_val_loss,
            }
        )

        top_checkpoints.sort(
            key=lambda item: (item["val_loss"], item["epoch"])
        )

        removed_checkpoints = []

        if len(top_checkpoints) > TOP_K_CHECKPOINTS:
            removed_checkpoints = top_checkpoints[
                TOP_K_CHECKPOINTS:
            ]
            top_checkpoints = top_checkpoints[
                :TOP_K_CHECKPOINTS
            ]

        current_top_epochs = {
            item["epoch"]
            for item in top_checkpoints
        }

        # ----------------------------------------------------
        # SAVE CURRENT EPOCH IF IT IS TOP-3
        # ----------------------------------------------------

        if epoch in current_top_epochs:

            epoch_checkpoint = dict(latest_checkpoint)
            epoch_checkpoint["checkpoint_rank"] = None

            epoch_path = (
                CHECKPOINT_DIR
                / f"gstcan_fold{fold_id}_epoch{epoch}.pth"
            )

            torch.save(
                epoch_checkpoint,
                epoch_path,
            )

        # ----------------------------------------------------
        # DELETE EPOCH CHECKPOINTS THAT LEFT TOP-3
        # ----------------------------------------------------

        for removed in removed_checkpoints:

            removed_epoch = int(
                removed["epoch"]
            )

            removed_path = (
                CHECKPOINT_DIR
                / f"gstcan_fold{fold_id}_epoch{removed_epoch}.pth"
            )

            if removed_path.exists():
                removed_path.unlink()

        # ----------------------------------------------------
        # REBUILD TOP-1 / TOP-2 / TOP-3 FILES
        # ----------------------------------------------------

        # Remove old ranked aliases first.
        for rank in range(1, TOP_K_CHECKPOINTS + 1):
            rank_path = (
                CHECKPOINT_DIR
                / f"gstcan_fold{fold_id}_top{rank}.pth"
            )
            if rank_path.exists():
                rank_path.unlink()

        for rank, item in enumerate(
            top_checkpoints,
            start=1,
        ):

            epoch_for_rank = int(
                item["epoch"]
            )

            source_path = (
                CHECKPOINT_DIR
                / f"gstcan_fold{fold_id}_epoch{epoch_for_rank}.pth"
            )

            if not source_path.exists():
                raise FileNotFoundError(
                    "Expected top checkpoint was not found: "
                    f"{source_path}"
                )

            rank_path = (
                CHECKPOINT_DIR
                / f"gstcan_fold{fold_id}_top{rank}.pth"
            )

            shutil.copy2(
                source_path,
                rank_path,
            )

            # Update the checkpoint rank and validation loss.
            ranked_checkpoint = torch.load(
                rank_path,
                map_location="cpu",
                weights_only=False,
            )

            ranked_checkpoint["checkpoint_rank"] = rank
            ranked_checkpoint["validation_loss"] = float(
                item["val_loss"]
            )

            torch.save(
                ranked_checkpoint,
                rank_path,
            )

        # ----------------------------------------------------
        # DELETE STALE EPOCH FILES
        # ----------------------------------------------------

        for epoch_path in CHECKPOINT_DIR.glob(
            f"gstcan_fold{fold_id}_epoch*.pth"
        ):

            match = re.search(
                r"_epoch(\d+)\.pth$",
                epoch_path.name,
            )

            if match is None:
                continue

            saved_epoch = int(
                match.group(1)
            )

            if saved_epoch not in current_top_epochs:
                epoch_path.unlink()

        # ----------------------------------------------------
        # PRINT EVERY EPOCH
        # ----------------------------------------------------

        top_text = ", ".join(
            f"E{item['epoch']}={item['val_loss']:.4f}"
            for item in top_checkpoints
        )

        print(
            f"Fold {fold_id} | "
            f"Epoch {epoch:03d}/{FINAL_EPOCHS} | "
            f"Train Loss={train_loss:.4f} | "
            f"Train Acc={train_accuracy:.4f} | "
            f"Val Loss={current_val_loss:.4f} | "
            f"Val Acc={val_metrics['accuracy']:.4f}"
        )

        print(
            f"   Top-{TOP_K_CHECKPOINTS} by Val Loss: "
            f"{top_text}"
        )

    # --------------------------------------------------------
    # TRAINING COMPLETE — REPORT TOP-3
    # --------------------------------------------------------

    print("\n----------------------------------------")
    print(
        f"FOLD {fold_id} TOP-{TOP_K_CHECKPOINTS} CHECKPOINTS"
    )
    print("----------------------------------------")

    assert len(top_checkpoints) == TOP_K_CHECKPOINTS, (
        f"Expected {TOP_K_CHECKPOINTS} checkpoints, "
        f"found {len(top_checkpoints)}"
    )

    for rank, item in enumerate(
        top_checkpoints,
        start=1,
    ):
        print(
            f"Top-{rank}: "
            f"Epoch {item['epoch']:03d} | "
            f"Val Loss = {item['val_loss']:.6f}"
        )

    # --------------------------------------------------------
    # LOAD BEST CHECKPOINT (TOP-1)
    # --------------------------------------------------------

    best_checkpoint_path = (
        CHECKPOINT_DIR
        / f"gstcan_fold{fold_id}_top1.pth"
    )

    assert best_checkpoint_path.exists(), (
        f"Best checkpoint not found: "
        f"{best_checkpoint_path}"
    )

    best_checkpoint = torch.load(
        best_checkpoint_path,
        map_location=DEVICE_TORCH,
        weights_only=False,
    )

    model.load_state_dict(
        best_checkpoint["model_state_dict"]
    )

    best_epoch = int(
        best_checkpoint["epoch"]
    )

    best_val_loss = float(
        best_checkpoint["validation_loss"]
    )

    print(
        f"\nUsing Top-1 checkpoint for final test evaluation: "
        f"Epoch {best_epoch:03d} | "
        f"Val Loss = {best_val_loss:.6f}"
    )

    # --------------------------------------------------------
    # BEST VALIDATION METRICS
    # --------------------------------------------------------

    best_history_index = best_epoch - 1

    best_val_accuracy = float(
        history["val_accuracy"][best_history_index]
    )

    best_val_precision = float(
        history["val_precision"][best_history_index]
    )

    best_val_recall = float(
        history["val_recall"][best_history_index]
    )

    best_val_f1 = float(
        history["val_f1"][best_history_index]
    )

    # --------------------------------------------------------
    # FINAL TEST EVALUATION
    # --------------------------------------------------------
    # Test set is evaluated ONLY with the Top-1 checkpoint.
    # It is never used for checkpoint selection.

    training_seconds = (
        time.perf_counter()
        - fold_start
    )

    final_test_metrics = evaluate_sequence_model(
        model,
        test_loader,
        DEVICE_TORCH,
        criterion,
    )

    # --------------------------------------------------------
    # SAVE FINAL CHECKPOINT
    # --------------------------------------------------------

    final_checkpoint = {
        "fold": fold_id,
        "epoch": best_epoch,
        "best_epoch": best_epoch,
        "best_validation_loss": best_val_loss,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history": history,

        "final_test_metrics":
            final_test_metrics,

        "top_checkpoints":
            top_checkpoints,

        "best_checkpoint_path":
            str(best_checkpoint_path),

        "train_sequences":
            train_sequences,

        "validation_sequences":
            val_sequences,

        "test_sequences":
            test_sequences,
    }

    torch.save(
        final_checkpoint,
        CHECKPOINT_DIR
        / f"gstcan_fold{fold_id}.pth",
    )

    # --------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------

    result = {
        "fold": fold_id,

        "train_sequences":
            train_sequences,

        "validation_sequences":
            val_sequences,

        "test_sequences":
            test_sequences,

        "training_seconds":
            training_seconds,

        "test_metrics":
            final_test_metrics,

        "history":
            history,

        "final_train_loss":
            history["train_loss"][-1],

        "final_train_accuracy":
            history["train_accuracy"][-1],

        "best_epoch":
            best_epoch,

        "best_val_loss":
            best_val_loss,

        "best_val_accuracy":
            best_val_accuracy,

        "best_val_precision":
            best_val_precision,

        "best_val_recall":
            best_val_recall,

        "best_val_f1":
            best_val_f1,

        "top_checkpoints":
            top_checkpoints,

        "best_checkpoint_path":
            str(best_checkpoint_path),

        # Kept for compatibility / reference to the final epoch.
        "final_val_loss":
            history["val_loss"][-1],

        "final_val_accuracy":
            history["val_accuracy"][-1],
    }

    fold_results.append(result)

    # --------------------------------------------------------
    # PRINT BEST VALIDATION
    # --------------------------------------------------------

    print("\n----------------------------------------")
    print(f"FOLD {fold_id} BEST VALIDATION")
    print("----------------------------------------")

    print(
        f"Best Epoch          : {best_epoch}"
    )

    print(
        f"Best Validation Loss: {best_val_loss:.6f}"
    )

    print(
        f"Validation Accuracy : {best_val_accuracy:.4f}"
    )

    print(
        f"Validation Precision: {best_val_precision:.4f}"
    )

    print(
        f"Validation Recall   : {best_val_recall:.4f}"
    )

    print(
        f"Validation F1       : {best_val_f1:.4f}"
    )

    # --------------------------------------------------------
    # PRINT FINAL TEST
    # --------------------------------------------------------

    print("\n----------------------------------------")
    print(
        f"FOLD {fold_id} TEST "
        f"(USING TOP-1 CHECKPOINT)"
    )
    print("----------------------------------------")

    print(
        f"Accuracy : "
        f"{final_test_metrics['accuracy']:.4f}"
    )

    print(
        f"Precision: "
        f"{final_test_metrics['precision']:.4f}"
    )

    print(
        f"Recall   : "
        f"{final_test_metrics['recall']:.4f}"
    )

    print(
        f"F1       : "
        f"{final_test_metrics['f1']:.4f}"
    )

    print(
        "Confusion matrix:\n",
        np.asarray(
            final_test_metrics[
                "confusion_matrix"
            ]
        ),
    )


# CELL 20 — SAVE ALL RESULTS
# ============================================================

results_file = RESULTS_DIR / "gstcan_3fold_results.pkl"
with open(results_file, "wb") as f:
    pickle.dump(
        {
            "fold_results": fold_results,
            "config": {
                "seed": SEED,
                "channels": GSTCAN_CHANNELS,
                "temporal_kernel": TEMPORAL_KERNEL,
                "dropout": DROPOUT,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "epochs": FINAL_EPOCHS,
                "validation_ratio": VALIDATION_RATIO,
                "num_folds": 3,
                "graph_operator": GRAPH_OPERATOR,
                "graph_gcn_projection": "single_1x1_after_fixed_shift",
            },
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )


# ============================================================


# ============================================================


# CELL 21 — BEST VALIDATION + TEST SUMMARY
# ============================================================

print("\n========================================")
print("3-FOLD FINAL RESULTS")
print("========================================")

# ------------------------------------------------------------
# BEST VALIDATION CHECKPOINT RESULTS
# ------------------------------------------------------------

best_val_metrics = [
    "best_val_loss",
    "best_val_accuracy",
]

for metric in best_val_metrics:

    values = np.asarray(
        [
            result[metric]
            for result in fold_results
        ],
        dtype=np.float64,
    )

    mean = values.mean()
    std = values.std(ddof=1)

    if metric == "best_val_loss":
        print(
            f"{metric:20s}: "
            f"{mean:.6f} ± {std:.6f}"
        )
    else:
        print(
            f"{metric:20s}: "
            f"{mean * 100:.2f}% ± {std * 100:.2f}%"
        )

print("\nBest checkpoint epochs:")

for result in fold_results:
    print(
        f"Fold {result['fold']}: "
        f"Epoch {result['best_epoch']} | "
        f"Val Loss = {result['best_val_loss']:.6f}"
    )

print("\nTop-3 checkpoints per fold:")

for result in fold_results:

    print(f"\nFold {result['fold']}")

    for rank, checkpoint in enumerate(
        result["top_checkpoints"],
        start=1,
    ):
        print(
            f"  Top-{rank}: "
            f"Epoch {checkpoint['epoch']:03d} | "
            f"Val Loss = {checkpoint['val_loss']:.6f}"
        )

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

test_metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
]

for metric in test_metrics:

    values = np.asarray(
        [
            result["test_metrics"][metric]
            for result in fold_results
        ],
        dtype=np.float64,
    )

    mean = values.mean()
    std = values.std(ddof=1)

    print(
        f"Test {metric:15s}: "
        f"{mean * 100:.2f}% ± "
        f"{std * 100:.2f}%"
    )


# CELL 22 — AGGREGATED CONFUSION MATRIX
# ============================================================

aggregated_cm = np.zeros((2, 2), dtype=np.int64)

for result in fold_results:
    aggregated_cm += np.asarray(
        result["test_metrics"]["confusion_matrix"],
        dtype=np.int64,
    )

TN, FP = aggregated_cm[0]
FN, TP = aggregated_cm[1]

print("\nAggregated 3-fold confusion matrix:")
print(aggregated_cm)
print(f"TN = {TN}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print(f"TP = {TP}")


# ============================================================


# ============================================================


# CELL 23 — PLOT TRAIN / VALIDATION CURVES
# ============================================================

for result in fold_results:

    fold_id = result["fold"]
    history = result["history"]

    epochs = np.arange(
        1,
        len(history["train_loss"]) + 1
    )

    # --------------------------------------------------------
    # LOSS
    # --------------------------------------------------------

    plt.figure(figsize=(10, 6))

    plt.plot(
        epochs,
        history["train_loss"],
        label="Train Loss",
    )

    plt.plot(
        epochs,
        history["val_loss"],
        label="Validation Loss",
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")

    plt.title(
        f"Fold {fold_id} — "
        "Training/Validation Loss"
    )

    plt.legend()
    plt.grid(
        True,
        alpha=0.3,
    )

    plt.show()

    # --------------------------------------------------------
    # ACCURACY
    # --------------------------------------------------------

    plt.figure(figsize=(10, 6))

    plt.plot(
        epochs,
        history["train_accuracy"],
        label="Train Accuracy",
    )

    plt.plot(
        epochs,
        history["val_accuracy"],
        label="Validation Accuracy",
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")

    plt.title(
        f"Fold {fold_id} — "
        "Training/Validation Accuracy"
    )

    plt.legend()
    plt.grid(
        True,
        alpha=0.3,
    )

    plt.show()


# CELL 24 — PLOT FINAL FOLD METRICS
# ============================================================

fold_ids = [result["fold"] for result in fold_results]

metrics = ["accuracy", "precision", "recall", "f1"]
metric_labels = ["Accuracy", "Precision", "Recall", "F1"]

values_by_metric = {
    metric: [
        result["test_metrics"][metric]
        for result in fold_results
    ]
    for metric in metrics
}

for metric, label in zip(metrics, metric_labels):
    values = np.asarray(values_by_metric[metric]) * 100.0

    plt.figure(figsize=(8, 5))
    plt.bar(fold_ids, values)
    plt.xlabel("Fold")
    plt.ylabel(f"{label} (%)")
    plt.title(f"3-Fold Final Test {label}")
    plt.xticks(fold_ids)
    plt.ylim(0, 100)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


# ============================================================


# CELL 25 — PLOT AGGREGATED CONFUSION MATRIX
# ============================================================

plt.figure(figsize=(6, 5))
plt.imshow(aggregated_cm, interpolation="nearest")
plt.title("Aggregated 3-Fold Final Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["ADL", "Fall"])
plt.yticks([0, 1], ["ADL", "Fall"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, int(aggregated_cm[i, j]), ha="center", va="center")

plt.colorbar()
plt.show()


# ============================================================


# CELL 26 — SAVE FINAL SUMMARY
# ============================================================

summary_file = RESULTS_DIR / "gstcan_final_summary.pkl"

final_summary = {
    "num_sequences": len(sequence_names),
    "num_folds": len(fold_results),
    "final_test_metrics_mean": {
        metric: float(
            np.mean(
                [
                    result["test_metrics"][metric]
                    for result in fold_results
                ]
            )
        )
        for metric in ["accuracy", "precision", "recall", "f1"]
    },
    "final_test_metrics_std": {
        metric: float(
            np.std(
                [
                    result["test_metrics"][metric]
                    for result in fold_results
                ],
                ddof=1,
            )
        )
        for metric in ["accuracy", "precision", "recall", "f1"]
    },
}

with open(summary_file, "wb") as f:
    pickle.dump(
        {
            "final_summary": final_summary,
            "aggregated_confusion_matrix": aggregated_cm.tolist(),
            "fold_results": fold_results,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print("\n========================================")
print("EXPERIMENT COMPLETE")
print("========================================")
print("Results:", results_file)
print("Summary:", summary_file)
print("Checkpoints:", CHECKPOINT_DIR)


# ============================================================
# CELL — EVALUATE ALL TOP-3 CHECKPOINTS ON TEST SET
# ============================================================
#
# For each fold:
#   Top-1 -> test set
#   Top-2 -> test set
#   Top-3 -> test set
#
# IMPORTANT:
#   - No training is performed here.
#   - No checkpoint selection is performed here.
#   - The test set is evaluated independently with each
#     previously saved Top-1/Top-2/Top-3 checkpoint.
#
# ============================================================

print("=" * 80)
print("TESTING ALL TOP-3 CHECKPOINTS")
print("=" * 80)


all_top3_test_results = []


for fold_info in folds:

    fold_id = fold_info["fold"]

    test_sequences = fold_info["test_sequences"]


    print()
    print("=" * 80)
    print(f"FOLD {fold_id}")
    print("=" * 80)


    # --------------------------------------------------------
    # TEST DATASET
    # --------------------------------------------------------

    test_dataset = ProcessedSequenceDataset(
        test_sequences,
        sequence_records,
    )


    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=full_sequence_collate,
    )


    # --------------------------------------------------------
    # CREATE MODEL
    # --------------------------------------------------------

    model = GSTCAN(
        adjacency_partitions=A_partition_tensor,
        input_channels=2,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT,
    ).to(DEVICE_TORCH)


    # --------------------------------------------------------
    # CHECK TOP-1 / TOP-2 / TOP-3
    # --------------------------------------------------------

    fold_top3_results = []


    for rank in range(
        1,
        TOP_K_CHECKPOINTS + 1,
    ):

        checkpoint_path = (
            CHECKPOINT_DIR
            / f"gstcan_fold{fold_id}_top{rank}.pth"
        )


        if not checkpoint_path.exists():

            raise FileNotFoundError(
                f"Checkpoint not found:\n"
                f"{checkpoint_path}"
            )


        print()
        print(
            "-" * 70
        )

        print(
            f"Evaluating Fold {fold_id} "
            f"Top-{rank}"
        )

        print(
            f"Checkpoint: "
            f"{checkpoint_path.name}"
        )


        # ----------------------------------------------------
        # LOAD CHECKPOINT
        # ----------------------------------------------------

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE_TORCH,
            weights_only=False,
        )


        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )


        model.eval()


        # ----------------------------------------------------
        # VALIDATION LOSS STORED IN CHECKPOINT
        # ----------------------------------------------------

        checkpoint_val_loss = float(
            checkpoint.get(
                "validation_loss",
                np.nan,
            )
        )


        checkpoint_epoch = int(
            checkpoint.get(
                "epoch",
                -1,
            )
        )


        # ----------------------------------------------------
        # TEST EVALUATION
        # ----------------------------------------------------

        test_metrics = evaluate_sequence_model(
            model,
            test_loader,
            DEVICE_TORCH,
            criterion,
        )


        # ----------------------------------------------------
        # STORE RESULTS
        # ----------------------------------------------------

        result = {

            "fold":
                fold_id,

            "checkpoint_rank":
                rank,

            "checkpoint_epoch":
                checkpoint_epoch,

            "checkpoint_validation_loss":
                checkpoint_val_loss,

            "checkpoint_path":
                str(
                    checkpoint_path
                ),

            "test_loss":
                test_metrics["loss"],

            "test_accuracy":
                test_metrics["accuracy"],

            "test_precision":
                test_metrics["precision"],

            "test_recall":
                test_metrics["recall"],

            "test_f1":
                test_metrics["f1"],

            "confusion_matrix":
                test_metrics[
                    "confusion_matrix"
                ],

            "test_samples":
                test_metrics[
                    "samples"
                ],

            "inference_seconds":
                test_metrics[
                    "inference_seconds"
                ],
        }


        fold_top3_results.append(
            result
        )

        all_top3_test_results.append(
            result
        )


        # ----------------------------------------------------
        # PRINT RESULT
        # ----------------------------------------------------

        print(
            f"Epoch          : "
            f"{checkpoint_epoch}"
        )

        print(
            f"Validation Loss: "
            f"{checkpoint_val_loss:.6f}"
        )

        print(
            f"Test Loss      : "
            f"{test_metrics['loss']:.6f}"
        )

        print(
            f"Test Accuracy  : "
            f"{test_metrics['accuracy']:.4f} "
            f"({test_metrics['accuracy'] * 100:.2f}%)"
        )

        print(
            f"Test Precision : "
            f"{test_metrics['precision']:.4f} "
            f"({test_metrics['precision'] * 100:.2f}%)"
        )

        print(
            f"Test Recall    : "
            f"{test_metrics['recall']:.4f} "
            f"({test_metrics['recall'] * 100:.2f}%)"
        )

        print(
            f"Test F1        : "
            f"{test_metrics['f1']:.4f} "
            f"({test_metrics['f1'] * 100:.2f}%)"
        )

        print(
            "Confusion Matrix:"
        )

        print(
            np.asarray(
                test_metrics[
                    "confusion_matrix"
                ]
            )
        )


# ============================================================
# SUMMARY TABLE
# ============================================================

print()
print("=" * 100)
print("ALL TOP-3 CHECKPOINT TEST RESULTS")
print("=" * 100)


print(
    f"{'Fold':<8}"
    f"{'Rank':<8}"
    f"{'Epoch':<10}"
    f"{'Val Loss':<14}"
    f"{'Test Loss':<14}"
    f"{'Accuracy':<12}"
    f"{'Precision':<12}"
    f"{'Recall':<12}"
    f"{'F1':<12}"
)


print("-" * 100)


for result in all_top3_test_results:

    print(
        f"{result['fold']:<8}"
        f"{result['checkpoint_rank']:<8}"
        f"{result['checkpoint_epoch']:<10}"
        f"{result['checkpoint_validation_loss']:<14.6f}"
        f"{result['test_loss']:<14.6f}"
        f"{result['test_accuracy'] * 100:<12.2f}"
        f"{result['test_precision'] * 100:<12.2f}"
        f"{result['test_recall'] * 100:<12.2f}"
        f"{result['test_f1'] * 100:<12.2f}"
    )


# ============================================================
# MEAN ± STD FOR EACH CHECKPOINT RANK
# ============================================================

print()
print("=" * 80)
print("AVERAGE TEST PERFORMANCE BY CHECKPOINT RANK")
print("=" * 80)


for rank in range(
    1,
    TOP_K_CHECKPOINTS + 1,
):

    rank_results = [
        result
        for result in all_top3_test_results
        if result[
            "checkpoint_rank"
        ] == rank
    ]


    accuracies = np.asarray(
        [
            result[
                "test_accuracy"
            ]
            for result in rank_results
        ],
        dtype=np.float64,
    )


    precisions = np.asarray(
        [
            result[
                "test_precision"
            ]
            for result in rank_results
        ],
        dtype=np.float64,
    )


    recalls = np.asarray(
        [
            result[
                "test_recall"
            ]
            for result in rank_results
        ],
        dtype=np.float64,
    )


    f1s = np.asarray(
        [
            result[
                "test_f1"
            ]
            for result in rank_results
        ],
        dtype=np.float64,
    )


    print()
    print(
        f"TOP-{rank}"
    )

    print(
        f"Accuracy : "
        f"{accuracies.mean() * 100:.2f}% ± "
        f"{accuracies.std(ddof=1) * 100:.2f}%"
    )

    print(
        f"Precision: "
        f"{precisions.mean() * 100:.2f}% ± "
        f"{precisions.std(ddof=1) * 100:.2f}%"
    )

    print(
        f"Recall   : "
        f"{recalls.mean() * 100:.2f}% ± "
        f"{recalls.std(ddof=1) * 100:.2f}%"
    )

    print(
        f"F1       : "
        f"{f1s.mean() * 100:.2f}% ± "
        f"{f1s.std(ddof=1) * 100:.2f}%"
    )


# ============================================================
# AGGREGATED CONFUSION MATRIX FOR EACH RANK
# ============================================================

print()
print("=" * 80)
print("AGGREGATED CONFUSION MATRICES")
print("=" * 80)


for rank in range(
    1,
    TOP_K_CHECKPOINTS + 1,
):

    aggregated_rank_cm = np.zeros(
        (
            2,
            2,
        ),
        dtype=np.int64,
    )


    for result in (
        all_top3_test_results
    ):

        if result[
            "checkpoint_rank"
        ] == rank:

            aggregated_rank_cm += (
                np.asarray(
                    result[
                        "confusion_matrix"
                    ],
                    dtype=np.int64,
                )
            )


    TN, FP = (
        aggregated_rank_cm[0]
    )

    FN, TP = (
        aggregated_rank_cm[1]
    )


    print()
    print(
        f"TOP-{rank} "
        "Aggregated Confusion Matrix:"
    )

    print(
        aggregated_rank_cm
    )

    print(
        f"TN = {TN}"
    )

    print(
        f"FP = {FP}"
    )

    print(
        f"FN = {FN}"
    )

    print(
        f"TP = {TP}"
    )


# ============================================================
# SAVE RESULTS
# ============================================================

top3_test_results_file = (
    RESULTS_DIR
    / "gstcan_all_top3_test_results.pkl"
)


with open(
    top3_test_results_file,
    "wb",
) as f:

    pickle.dump(
        all_top3_test_results,
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )


print()
print("=" * 80)
print("TOP-3 TEST EVALUATION COMPLETE")
print("=" * 80)

print(
    "Saved results to:"
)

print(
    top3_test_results_file
)



Python       : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch      : 2.11.0+cu128
PyTorch CUDA : 12.8
CUDA         : True
GPU          : Tesla T4
Processed directory: /content/drive/MyDrive/UrProcessed
Environment and path verification PASSED.
PROCESSED DATASET DISCOVERY
Processed directory: /content/drive/MyDrive/UrProcessed
PKL files found    : 257
Fall sequences     : 158
ADL sequences      : 99
adl-01                                   | ADL   | frames=   146 | /content/drive/MyDrive/UrProcessed/Copy of adl-01.pkl
adl-02                                   | ADL   | frames=   152 | /content/drive/MyDrive/UrProcessed/Copy of adl-02.pkl
adl-03                                   | ADL   | frames=   180 | /content/drive/MyDrive/UrProcessed/Copy of adl-03.pkl
adl-04                                   | ADL   | frames=   150 | /content/drive/MyDrive/UrProcessed/Copy of adl-04.pkl
adl-05                                   | ADL   | frames=   180 | /content/drive/MyDrive/UrProcessed/